# Confidence $\varrho$

The confidence $\varrho$ is read from the attribution process. This notebook constructs it on the oracle of \textcite{rosa-2021}, where the correct defective change is known, and checks that it orders attributions by correctness.

## Load

Both procedures are streamed once. Each run yields its named defective change and the signals its trajectory exposes: reasoning volume, number of inspections, reasoning variance, tool failures, inspected diff length, and whether it inspected the change it named.

In [1]:
import json, re
from collections import defaultdict

import numpy as np
import pyarrow as pa
import polars as pl
import pygit2
import plotly.graph_objects as go
from scipy.stats import rankdata
from tqdm.auto import tqdm
from benchmark_notebook import theme  # noqa: F401
from benchmark_notebook.io import CloudStorage
from benchmark.attribution.history import VersionControlPathWalker

storage = CloudStorage(bucket="bug-request-bucket")
models = {
    None: "MiniMax M2.7",
    "minimax-m3": "MiniMax M3",
    "deepseek-v4-pro": "DeepSeek V4 Pro",
}


def load():
    for procedure, key, version in (
        ("AgentSZZ", "attribution-agent-szz-evaluation", "1.0.1"),
        ("AgenticSZZ", "attribution-agentic-szz-evaluation", "1.0.0"),
    ):
        steps, patch, shown, fails, named, meta = (
            defaultdict(list),
            defaultdict(int),
            defaultdict(set),
            defaultdict(int),
            {},
            {},
        )
        for batch in tqdm(
            storage.asset(
                key=key,
                version=version,
                location="main",
                partition="default",
                columns=[
                    "trace",
                    "repository",
                    "correction",
                    "node",
                    "parent",
                    "event",
                    "payload",
                    "partition",
                ],
                batch=20,
            ),
            desc=procedure,
            unit="batch",
        ):
            for row in (
                pl.from_arrow(pa.Table.from_batches([batch]))
                .with_columns(pl.col("partition").cast(pl.String))
                .iter_rows(named=True)
            ):
                meta[row["trace"]] = row
                if row["event"] == "tool_error":
                    fails[row["trace"]] += 1
                if not row["payload"]:
                    continue
                event, node, data = (
                    row["event"],
                    row["node"],
                    json.loads(row["payload"]),
                )
                if event == "llm_end":
                    steps[row["trace"]].append(
                        ((data.get("llm_output") or {}).get("token_usage") or {}).get(
                            "completion_tokens"
                        )
                        or 0
                    )
                elif (
                    event == "tool_end"
                    and node == "show"
                    and isinstance((data.get("output") or {}).get("content"), dict)
                ):
                    patch[row["trace"]] += len(
                        data["output"]["content"].get("patch") or ""
                    )
                elif (
                    event == "tool_start"
                    and node == "show"
                    and data.get("inputs", {}).get("commit")
                ):
                    shown[row["trace"]].add(data["inputs"]["commit"].lower())
                elif (
                    node == "LangGraph"
                    and event == "node_end"
                    and row["parent"] is None
                ):
                    named[row["trace"]] = (
                        (data.get("outputs") or {}).get("introducer") or ""
                    ).lower() or None
        for trace, row in meta.items():
            token = (
                row["partition"].split("|", 1)[1].partition("-")[2]
                if "|" in row["partition"]
                else ""
            )
            pred = named.get(trace)
            yield {
                "procedure": procedure,
                "model": models.get(token or None, token),
                "repo_id": row["partition"].split("|", 1)[0],
                "repository": row["repository"],
                "correction": row["correction"],
                "pred": pred,
                "reasoning_volume": sum(steps[trace]),
                "inspections": len(shown[trace]),
                "reasoning_variance": (
                    float(np.var(steps[trace])) if len(steps[trace]) > 1 else 0.0
                ),
                "tool_failures": fails[trace],
                "inspected_diff": patch[trace],
                "inspected_named": int(
                    pred is not None
                    and any(
                        s.startswith(pred[:12]) or pred.startswith(s[:12])
                        for s in shown[trace]
                    )
                ),
            }


runs = pl.DataFrame(load())
print(runs.group_by(["procedure", "model"]).len().sort(["procedure", "model"]))

AgentSZZ: 0batch [00:00, ?batch/s]

AgenticSZZ: 0batch [00:00, ?batch/s]

shape: (6, 3)
┌────────────┬─────────────────┬─────┐
│ procedure  ┆ model           ┆ len │
│ ---        ┆ ---             ┆ --- │
│ str        ┆ str             ┆ u32 │
╞════════════╪═════════════════╪═════╡
│ AgentSZZ   ┆ DeepSeek V4 Pro ┆ 756 │
│ AgentSZZ   ┆ MiniMax M2.7    ┆ 756 │
│ AgentSZZ   ┆ MiniMax M3      ┆ 756 │
│ AgenticSZZ ┆ DeepSeek V4 Pro ┆ 756 │
│ AgenticSZZ ┆ MiniMax M2.7    ┆ 756 │
│ AgenticSZZ ┆ MiniMax M3      ┆ 756 │
└────────────┴─────────────────┴─────┘


## Repositories

The named-change properties (temporal distance, diff size, file overlap, file history) and the error check read the git history directly, so the source repositories are fetched once from the validation-sources asset and extracted under `/tmp/repos`. Partitions already extracted are skipped.

In [2]:
import tarfile
from pathlib import Path
from google.cloud import storage as gcs

base = Path("/tmp/repos")
base.mkdir(exist_ok=True)
bucket = gcs.Client(project="bug-request").bucket("bug-request-bucket")
sources = {}
for blob in bucket.list_blobs(
    prefix="assets/ingest-validation-sources/version=1.0.1/location=main/"
):
    if blob.name.endswith(".tar.gz"):
        partition = next(
            s.replace("partition=", "")
            for s in blob.name.split("/")
            if s.startswith("partition=")
        )
        sources.setdefault(partition, blob)

for partition, blob in tqdm(sources.items(), desc="repositories", unit="repo"):
    target = base / partition
    if (target / ".done").exists():
        continue
    target.mkdir(exist_ok=True)
    archive = target / "r.tar.gz"
    blob.download_to_filename(str(archive))
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(target, filter="data")
    archive.unlink()
    (target / ".done").touch()
print("repositories ready:", sum(1 for _ in base.glob("*/.done")))

repositories:   0%|          | 0/351 [00:00<?, ?repo/s]

repositories ready: 351


## Ground truth

The oracle index gives the confirmed defective changes per corrective change, the BIC category, and the language. Correctness `y` is fixed independently of $\varrho$.

In [3]:
index = pl.from_arrow(
    storage.asset(
        key="ingest-validation-context-index",
        version="2.0.0",
        location="main",
        partition="default",
    )
)

runs = (
    runs.join(
        index.select(["repository", "correction", "defects"]),
        on=["repository", "correction"],
        how="left",
    )
    .join(
        index.select(["repository", "correction", "defects", "commits"])
        .explode("defects")
        .explode("commits")
        .drop_nulls("commits")
        .unnest("commits")
        .filter(pl.col("commit") == pl.col("defects"))
        .group_by(["repository", "correction"])
        .agg(pl.col("category").first().str.replace_all("_", "-")),
        on=["repository", "correction"],
        how="left",
    )
    .join(
        index.select(["repository", "correction", "languages"])
        .with_columns(
            pl.when(
                pl.col("languages").list.contains("py")
                & pl.col("languages").list.contains("js")
            )
            .then(pl.lit("py-js"))
            .when(pl.col("languages").list.contains("py"))
            .then(pl.lit("py"))
            .when(pl.col("languages").list.contains("js"))
            .then(pl.lit("js"))
            .otherwise(pl.lit("other"))
            .alias("language")
        )
        .select(["repository", "correction", "language"]),
        on=["repository", "correction"],
        how="left",
    )
    .with_columns(
        pl.col("category").fill_null("outside"),
        pl.col("defects").list.eval(pl.element().str.to_lowercase()).alias("defects"),
    )
)

runs = runs.with_columns(
    pl.struct(["pred", "defects"])
    .map_elements(
        lambda s: int(
            s["pred"] is not None
            and any(s["pred"][:12] == d[:12] for d in (s["defects"] or []))
        ),
        return_dtype=pl.Int64,
    )
    .alias("y")
)
del index
print(
    "runs:",
    runs.height,
    "| accuracy by procedure:",
    runs.group_by("procedure")
    .agg(pl.col("y").mean().round(3))
    .sort("procedure")
    .to_dicts(),
)

runs: 4536 | accuracy by procedure: [{'procedure': 'AgentSZZ', 'y': 0.714}, {'procedure': 'AgenticSZZ', 'y': 0.7}]


## Errors

A prediction that is not a valid SHA, or names a commit absent from the repository, is an error. Errors are excluded from $\varrho$ but enter the F1 recall denominator as a miss, following the replication packages.

In [4]:
repos, errors = {}, {}
for triple in (
    runs.select(["repo_id", "correction", "pred"]).unique().iter_rows(named=True)
):
    pred = triple["pred"]
    if pred is None or not re.fullmatch(r"[0-9a-f]{7,40}", pred):
        errors[tuple(triple.values())] = True
        continue
    try:
        repo = repos.get(triple["repo_id"]) or pygit2.Repository(
            f"/tmp/repos/{triple['repo_id']}/repository.git"
        )
        repos[triple["repo_id"]] = repo
        repo.revparse_single(pred)
        errors[tuple(triple.values())] = False
    except Exception:
        errors[tuple(triple.values())] = True

runs = runs.with_columns(
    pl.struct(["repo_id", "correction", "pred"])
    .map_elements(
        lambda s: errors[(s["repo_id"], s["correction"], s["pred"])],
        return_dtype=pl.Boolean,
    )
    .alias("error")
)
print("errors:", runs.filter(pl.col("error")).height, "of", runs.height)

errors: 387 of 4536


## Procedure comparison

Both procedures are scored by category and model in F1 (micro per run, averaged over runs) on the Python subset. Blame saturates for both; on the harder categories the procedures differ in how their search responds to difficulty. This decides which procedure carries $\varrho$.

In [5]:
bic = {
    "blame": "Blame",
    "correction-ancestor": "Correction ancestor",
    "blame-ancestor": "Blame ancestor",
    "outside": "Unreachable",
}

comparison = (
    runs.filter(pl.col("language") == "py")
    .with_columns(
        pl.col("correction")
        .cum_count()
        .over(["procedure", "model", "correction"])
        .alias("run"),
        pl.col("category").replace(bic).alias("cat"),
    )
    .group_by(["procedure", "model", "cat", "run"])
    .agg(
        pl.col("y").sum().alias("tp"),
        (~pl.col("error")).sum().alias("answered"),
        pl.len().alias("changes"),
    )
    .with_columns(
        (pl.col("tp") / pl.col("answered")).alias("p"),
        (pl.col("tp") / pl.col("changes")).alias("r"),
    )
    .with_columns(
        pl.when(pl.col("p") + pl.col("r") > 0)
        .then(2 * pl.col("p") * pl.col("r") / (pl.col("p") + pl.col("r")))
        .otherwise(0.0)
        .alias("f1")
    )
    .group_by(["procedure", "cat", "model"])
    .agg(pl.col("f1").mean().round(2).alias("f1"))
)
comparison.pivot(values="f1", index=["procedure", "cat"], on="model")

procedure,cat,DeepSeek V4 Pro,MiniMax M2.7,MiniMax M3
str,str,f64,f64,f64
"""AgenticSZZ""","""Blame""",0.92,0.85,0.91
"""AgentSZZ""","""Unreachable""",0.26,0.14,0.31
"""AgenticSZZ""","""Blame ancestor""",0.29,0.21,0.36
"""AgentSZZ""","""Correction ancestor""",0.48,0.37,0.48
"""AgenticSZZ""","""Unreachable""",0.0,0.0,0.0
"""AgentSZZ""","""Blame""",0.88,0.82,0.89
"""AgenticSZZ""","""Correction ancestor""",0.29,0.22,0.3
"""AgentSZZ""","""Blame ancestor""",0.3,0.0,0.18


## Selection across runs

After choosing the procedure, the released prediction for a corrective change is selected across its runs by majority vote, and the vote share is read as a first reliability signal. Isolated from any feature, this asks only whether agreement across runs carries information. Correctness is the majority SHA against the confirmed inducers of the corrective change.

In [6]:
from collections import Counter
from sklearn.metrics import roc_auc_score


def majority(frame):
    rows = []
    for keys, group in frame.filter(
        (~pl.col("error")) & pl.col("pred").is_not_null() & (pl.col("language") == "py")
    ).group_by(["procedure", "repository", "correction", "model"]):
        sha, votes = Counter(group["pred"].to_list()).most_common(1)[0]
        gold = group["defects"].first() or []
        rows.append(
            {
                "procedure": keys[0],
                "model": keys[2],
                "vote": votes / group.height,
                "Y": int(any(sha[:12] == d[:12] for d in gold)),
            }
        )
    return pl.DataFrame(rows)


vote = majority(runs)
report = []
for scope in ("AgentSZZ", "AgenticSZZ", "pooled"):
    part = vote if scope == "pooled" else vote.filter(pl.col("procedure") == scope)
    agree = part.filter(pl.col("vote") == 1.0)
    disagree = part.filter(pl.col("vote") < 1.0)
    report.append(
        {
            "scope": scope,
            "AUROC": round(roc_auc_score(part["Y"], part["vote"]), 3),
            "base": round(part["Y"].mean(), 2),
            "agree acc": round(agree["Y"].mean(), 2),
            "disagree acc": round(disagree["Y"].mean(), 2),
            "n": part.height,
        }
    )
pl.DataFrame(report)

scope,AUROC,base,agree acc,disagree acc,n
str,f64,f64,f64,f64,i64
"""AgentSZZ""",0.654,0.77,0.83,0.51,733
"""AgenticSZZ""",0.623,0.76,0.81,0.49,715
"""pooled""",0.638,0.76,0.82,0.5,1448


## Candidate signals

The confidence draws on signals the attribution process exposes. We list the candidates by source and record which we discard before the pre-study and why. The surviving candidates pass a consistency and a redundancy check; selection uses no weights or calibration, only rank normalization.

| Source | Signal | Status |
|---|---|---|
| Model-internal (single pass) | Verbalized self-confidence | excluded, systematically overconfident \cite{xiong-2024, shi-2026}, not solicited by design |
| | Token probabilities | excluded, not emitted by the procedure under the standard run |
| Search trajectory | Reasoning volume | candidate |
| | Number of inspections | candidate |
| | Reasoning variance | candidate |
| | Tool failures | candidate |
| | Self-correction | excluded, the procedure carries no self-correction step |
| | Inspection of the named change | candidate |
| | Inspected diff length | candidate |
| Agreement | Vote share across runs | candidate, the selected prediction carries its agreement |
| Named-change properties | Temporal distance to the corrective change | candidate |
| | Diff size of the named change | candidate |
| | Number of files changed | candidate |
| | File overlap with the corrective change | candidate |
| | File-history depth | candidate, ghost commits fall back to temporal distance |

AgentSZZ adapts its search to the case, so its process tracks difficulty. The confidence is built on it. For each change the majority prediction is selected across runs, and the candidate signals are taken on it: the trajectory signals averaged over the runs that named it, the named-change properties from the repository, and the vote share as agreement.

In [7]:
agent = runs.filter(
    (pl.col("procedure") == "AgentSZZ")
    & (~pl.col("error"))
    & pl.col("pred").is_not_null()
    & (pl.col("language") == "py")
)


def file_history(repo, corrective, target, files):
    best = None
    for path in files:
        try:
            walker = VersionControlPathWalker(
                repo, str(corrective.id), path, depth=2000
            )
            for i, entry in enumerate(walker.entries()):
                if str(repo.revparse_single(entry.commit).id) == target:
                    best = i if best is None else min(best, i)
                    break
        except Exception:
            continue
    return best


def properties(repo_id, correction, pred):
    repo = repos[repo_id]
    corrective, named = repo.revparse_single(correction), repo.revparse_single(pred)
    age = abs(corrective.commit_time - named.commit_time) / 86400.0
    named_diff = repo.diff(named.parents[0], named) if named.parents else None
    corrective_files = (
        {d.new_file.path for d in repo.diff(corrective.parents[0], corrective).deltas}
        if corrective.parents
        else set()
    )
    named_files = {d.new_file.path for d in named_diff.deltas} if named_diff else set()
    diff_size = (
        sum(h.old_lines + h.new_lines for d in (named_diff or []) for h in d.hunks)
        if named_diff
        else 0
    )
    overlap = (
        len(named_files & corrective_files) / len(corrective_files)
        if corrective_files
        else 0.0
    )
    # file-history depth of the prediction in the corrective files; ghost commits
    # are unreachable along that history, so they fall back to temporal distance
    depth = file_history(repo, corrective, str(named.id), corrective_files)
    return (
        age,
        diff_size,
        len(named_files),
        overlap,
        float(depth) if depth is not None else age,
    )


geometry = {
    t: properties(*t)
    for t in tqdm(
        [
            tuple(r.values())
            for r in agent.select(["repo_id", "correction", "pred"])
            .unique()
            .iter_rows(named=True)
        ],
        desc="properties",
        unit="pred",
    )
}
agent = agent.with_columns(
    pl.struct(["repo_id", "correction", "pred"])
    .map_elements(
        lambda s: geometry[(s["repo_id"], s["correction"], s["pred"])][0],
        return_dtype=pl.Float64,
    )
    .alias("temporal_distance"),
    pl.struct(["repo_id", "correction", "pred"])
    .map_elements(
        lambda s: geometry[(s["repo_id"], s["correction"], s["pred"])][1],
        return_dtype=pl.Int64,
    )
    .alias("diff_size"),
    pl.struct(["repo_id", "correction", "pred"])
    .map_elements(
        lambda s: geometry[(s["repo_id"], s["correction"], s["pred"])][2],
        return_dtype=pl.Int64,
    )
    .alias("files_changed"),
    pl.struct(["repo_id", "correction", "pred"])
    .map_elements(
        lambda s: geometry[(s["repo_id"], s["correction"], s["pred"])][3],
        return_dtype=pl.Float64,
    )
    .alias("file_overlap"),
    pl.struct(["repo_id", "correction", "pred"])
    .map_elements(
        lambda s: geometry[(s["repo_id"], s["correction"], s["pred"])][4],
        return_dtype=pl.Float64,
    )
    .alias("file_history"),
)

# one row per change: the majority SHA, its signals averaged over the runs that named it, the vote share, gold correctness
trajectory = (
    "reasoning_volume",
    "inspections",
    "reasoning_variance",
    "tool_failures",
    "inspected_diff",
)
geometry_signals = (
    "temporal_distance",
    "diff_size",
    "files_changed",
    "file_overlap",
    "file_history",
)
records = []
for keys, group in agent.group_by(["repository", "correction", "model"]):
    sha, votes = Counter(group["pred"].to_list()).most_common(1)[0]
    chosen = group.filter(pl.col("pred") == sha)
    gold = group["defects"].first() or []
    record = {
        "model": keys[2],
        "vote": votes / group.height,
        "category": group["category"].first(),
        "Y": int(any(sha[:12] == d[:12] for d in gold)),
    }
    for signal in trajectory + geometry_signals:
        record[signal] = chosen[signal].mean()
    records.append(record)
selected = pl.DataFrame(records)
print("changes:", selected.height, "| base rate:", round(selected["Y"].mean(), 3))

properties:   0%|          | 0/406 [00:00<?, ?pred/s]

changes: 733 | base rate: 0.767


## Consistency and redundancy

Each candidate signal is taken on the majority-selected change and checked for consistency, whether it orders correctness on its own, and for redundancy, whether it duplicates another. Consistency is the univariate AUROC per model, oriented so the value is the ordering strength and the direction is reported separately; redundancy is the Spearman correlation, with $0.70$ as the threshold. No combination is optimized on the oracle; the checks only filter.

In [8]:
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

features = [
    "vote",
    "reasoning_volume",
    "inspections",
    "reasoning_variance",
    "tool_failures",
    "inspected_diff",
    "temporal_distance",
    "diff_size",
    "files_changed",
    "file_overlap",
    "file_history",
]
frame = selected.to_pandas()


def consistency(feature):
    raw = {
        m: roc_auc_score(
            frame[frame["model"] == m]["Y"], frame[frame["model"] == m][feature]
        )
        for m in ("MiniMax M2.7", "MiniMax M3", "DeepSeek V4 Pro")
    }
    oriented = {m: max(a, 1 - a) for m, a in raw.items()}
    return {
        "signal": feature,
        "M2.7": round(oriented["MiniMax M2.7"], 2),
        "M3": round(oriented["MiniMax M3"], 2),
        "V4 Pro": round(oriented["DeepSeek V4 Pro"], 2),
        "power": round(float(np.median(list(oriented.values()))), 2),
        "direction": "correct" if np.mean(list(raw.values())) > 0.5 else "wrong",
    }


with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=60):
    display(
        pl.DataFrame([consistency(f) for f in features]).sort("power", descending=True)
    )

signal,M2.7,M3,V4 Pro,power,direction
str,f64,f64,f64,f64,str
"""temporal_distance""",0.75,0.78,0.73,0.75,"""wrong"""
"""file_history""",0.73,0.78,0.75,0.75,"""wrong"""
"""inspected_diff""",0.63,0.71,0.68,0.68,"""wrong"""
"""diff_size""",0.63,0.68,0.68,0.68,"""wrong"""
"""vote""",0.64,0.66,0.66,0.66,"""correct"""
"""reasoning_volume""",0.65,0.66,0.69,0.66,"""wrong"""
"""inspections""",0.58,0.62,0.6,0.6,"""wrong"""
"""files_changed""",0.56,0.61,0.59,0.59,"""wrong"""
"""file_overlap""",0.62,0.58,0.58,0.58,"""correct"""


In [9]:
import itertools

with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=60):
    display(
        pl.DataFrame(
            [
                {
                    "signal A": a,
                    "signal B": b,
                    "spearman": round(spearmanr(frame[a], frame[b]).correlation, 2),
                }
                for a, b in itertools.combinations(features, 2)
                if abs(spearmanr(frame[a], frame[b]).correlation) >= 0.70
            ]
        )
    )

signal A,signal B,spearman
str,str,f64
"""diff_size""","""files_changed""",0.71


### Diagnostic: signals by gold class

The pooled view is dominated by the blame majority. We facet by the gold BIC class to see whether a signal separates correct from wrong within a class. The gold class is leakage and serves analysis only; it is never available to $\varrho$.

In [10]:
from plotly.subplots import make_subplots
from scipy.stats import rankdata

gold = {
    "blame": "Blame",
    "correction-ancestor": "Corr. anc.",
    "blame-ancestor": "Blame anc.",
    "outside": "Unreach.",
}
classes = list(gold)
diagnostic = make_subplots(
    rows=len(features),
    cols=len(classes),
    row_titles=features,
    column_titles=[gold[c] for c in classes],
    shared_xaxes=True,
)
for r, feature in enumerate(features):
    rank = (rankdata(frame[feature].values) - 1) / (len(frame) - 1)
    for col, klass in enumerate(classes):
        mask = frame["category"].values == klass
        for label, color in ((1, "#2c7fb8"), (0, "#d95f0e")):
            diagnostic.add_trace(
                go.Histogram(
                    x=rank[mask & (frame["Y"].values == label)],
                    nbinsx=15,
                    marker_color=color,
                    opacity=0.6,
                    showlegend=False,
                ),
                row=r + 1,
                col=col + 1,
            )
diagnostic.update_layout(
    height=1700,
    width=900,
    barmode="overlay",
    title="Rank-normalized signal by gold class (blue correct, orange wrong)",
    template="simple_white",
)
diagnostic.update_annotations(font_size=10)
diagnostic.show()

## Confidence $\varrho$

The released prediction is the majority change across runs, and its reliability $\rho$ combines the surviving signals: the vote share (high implies correct) and the temporal distance of the prediction to the corrective change (high implies wrong). Each is rank normalized within a model and combined by its direction, with no weights or calibration. The candidate aggregation $\rho(c) = \max$ is trivial here, since majority selection already yields one prediction per change; it takes effect on the corpus, where a change carries several candidate corrective relations.

In [11]:
inverse = ("temporal_distance",)  # high implies wrong, inverted
direct = ("vote",)  # high implies correct, used as is

parts = []
for subset in (
    selected.filter(pl.col("model") == m).to_pandas()
    for m in selected["model"].unique().to_list()
):
    n = len(subset)
    terms = [1 - (rankdata(subset[s].values) - 1) / (n - 1) for s in inverse]
    terms += [(rankdata(subset[s].values) - 1) / (n - 1) for s in direct]
    subset["rho"] = sum(terms) / len(terms)
    parts.append(pl.from_pandas(subset))
changes = pl.concat(parts)
print("changes:", changes.height, "| base rate:", round(changes["Y"].mean(), 3))

changes: 733 | base rate: 0.767


## Ordering on the oracle

Each change is ranked by its reliability $\rho$ against gold correctness `Y`, the majority prediction matched to the confirmed inducers of the corrective change. AUROC measures whether high $\rho$ separates correct from wrong. `base` is the unconditional accuracy of the majority prediction, the share correct over all changes; risk-coverage then reads how accuracy on the most confident fraction rises above that base.

In [12]:
class Ranking:
    """Tie-handled AUROC over the change-level reliability rho against gold correctness."""

    def _ranks(self, values):
        order = np.argsort(values, kind="mergesort")
        ranks = np.empty(len(values))
        i = 0
        while i < len(values):
            j = i
            while j + 1 < len(values) and values[order[j + 1]] == values[order[i]]:
                j += 1
            ranks[order[i : j + 1]] = (i + j) / 2.0 + 1.0
            i = j + 1
        return ranks

    def auroc(self, scores, labels):
        positives, total = int(labels.sum()), len(labels)
        if positives in (0, total):
            return float("nan")
        return (
            self._ranks(scores)[labels == 1].sum() - positives * (positives + 1) / 2
        ) / (positives * (total - positives))


ranking = Ranking()
print(
    pl.DataFrame(
        [
            {
                "model": m,
                "AUROC": round(
                    ranking.auroc(
                        changes.filter(pl.col("model") == m)["rho"].to_numpy(),
                        changes.filter(pl.col("model") == m)["Y"].to_numpy(),
                    ),
                    3,
                ),
                "base": round(changes.filter(pl.col("model") == m)["Y"].mean(), 2),
                "n": changes.filter(pl.col("model") == m).height,
            }
            for m in sorted(changes["model"].unique().to_list())
        ]
    )
)

shape: (3, 4)
┌─────────────────┬───────┬──────┬─────┐
│ model           ┆ AUROC ┆ base ┆ n   │
│ ---             ┆ ---   ┆ ---  ┆ --- │
│ str             ┆ f64   ┆ f64  ┆ i64 │
╞═════════════════╪═══════╪══════╪═════╡
│ DeepSeek V4 Pro ┆ 0.767 ┆ 0.79 ┆ 246 │
│ MiniMax M2.7    ┆ 0.775 ┆ 0.73 ┆ 245 │
│ MiniMax M3      ┆ 0.818 ┆ 0.78 ┆ 242 │
└─────────────────┴───────┴──────┴─────┘


In [13]:
figure = go.Figure()
for m in sorted(changes["model"].unique().to_list()):
    rho = changes.filter(pl.col("model") == m)["rho"].to_numpy()
    correct = changes.filter(pl.col("model") == m)["Y"].to_numpy()
    coverage = [(rho >= t).mean() for t in sorted(set(rho), reverse=True)]
    accuracy = [correct[rho >= t].mean() for t in sorted(set(rho), reverse=True)]
    figure.add_trace(go.Scatter(x=coverage, y=accuracy, mode="lines", name=m))
figure.add_hline(y=changes["Y"].mean(), line=dict(color="#808080", dash="dot"))
figure.update_layout(
    height=360,
    width=560,
    xaxis_title="coverage",
    yaxis_title="accuracy on covered changes",
    template="simple_white",
)
figure.show()

## Combination audit

The confidence uses two signals, the vote share and the temporal distance. This audit asks whether a third signal earns its place. Each remaining candidate is added to the `vote + age` anchor under the same rank normalization, and we read the resulting AUROC per model. A signal is admitted only if it raises the ordering across all three models beyond the run-to-run noise, taken here as $0.01$ AUROC. None does: the best single addition moves the mean by $0.01$ and helps one model while it costs another, so $\varrho$ stays at the two signals that each carry an independent rationale, agreement from the selection and topological distance from reachability. Parsimony is preferred over an unstable gain.

In [14]:
class Combination:
    """AUROC of a rank-normalized signal set on the majority-selected change, per model."""

    def __init__(self, table, ranker):
        self.table = table
        self.ranker = ranker
        self.scale = sorted(table["model"].unique().to_list())

    def _score(self, subset, inverse, direct):
        n = len(subset)
        terms = [1 - (rankdata(subset[s].values) - 1) / (n - 1) for s in inverse]
        terms += [(rankdata(subset[s].values) - 1) / (n - 1) for s in direct]
        return sum(terms) / len(terms)

    def evaluate(self, inverse, direct):
        row = {}
        scores = []
        for m in self.scale:
            subset = self.table.filter(pl.col("model") == m).to_pandas()
            value = self.ranker.auroc(
                self._score(subset, inverse, direct), subset["Y"].values
            )
            row[m] = round(value, 3)
            scores.append(value)
        row["worst"] = round(float(np.min(scores)), 3)
        return row


audit = Combination(selected, ranking)
candidates = {
    "vote + age": (("temporal_distance",), ("vote",)),
    "+ inspected_diff": (("temporal_distance", "inspected_diff"), ("vote",)),
    "+ reasoning_volume": (("temporal_distance", "reasoning_volume"), ("vote",)),
    "+ diff_size": (("temporal_distance", "diff_size"), ("vote",)),
    "+ inspections": (("temporal_distance", "inspections"), ("vote",)),
    "+ files_changed": (("temporal_distance", "files_changed"), ("vote",)),
    "+ file_overlap": (("temporal_distance",), ("vote", "file_overlap")),
    "age only": (("temporal_distance",), ()),
    "vote only": ((), ("vote",)),
}
with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=60):
    display(
        pl.DataFrame(
            [
                {"set": name, **audit.evaluate(inv, dir)}
                for name, (inv, dir) in candidates.items()
            ]
        )
    )

set,DeepSeek V4 Pro,MiniMax M2.7,MiniMax M3,worst
str,f64,f64,f64,f64
"""vote + age""",0.767,0.775,0.818,0.767
"""+ inspected_diff""",0.778,0.761,0.831,0.761
"""+ reasoning_volume""",0.786,0.769,0.817,0.769
"""+ diff_size""",0.775,0.768,0.817,0.768
"""+ inspections""",0.751,0.74,0.816,0.74
"""+ files_changed""",0.744,0.748,0.788,0.744
"""+ file_overlap""",0.785,0.792,0.814,0.785
"""age only""",0.725,0.746,0.781,0.725
"""vote only""",0.661,0.641,0.659,0.641


## Temperature ablation

The vote share reads agreement across runs, yet the main runs decode greedily, where the three samples vary only through residual nondeterminism \cite{yuan-2026}. This ablation asks whether genuine sampling diversity sharpens the agreement signal. MiniMax M3 was rerun three times at temperature $0.7$, the canonical self-consistency setting \cite{wang-2023}, on the same corrective changes, and is compared against its greedy runs. We read how often the runs agree, the accuracy of the agreeing and the split cases, and the vote AUROC with a bootstrap interval. The hot runs carry token logprobs, so they are read per partition blob rather than streamed at once.

In [ ]:
import io
import gc
import pyarrow.parquet as pq
from concurrent.futures import ThreadPoolExecutor


class TemperatureArm:
    """Loads one decoding arm of the AgentSZZ M3 runs by reading partition blobs in parallel.

    The Dagster partition (repository, run, model suffix) lives in the blob path, not in the
    parquet, and the hot runs carry token logprobs, so the whole-asset stream is too large.
    Only the final graph output row is parsed and the heavy table is released per blob.
    """

    columns = ["trace", "repository", "correction", "node", "parent", "event", "payload"]

    def __init__(self, version, suffix_match):
        self.version = version
        self.suffix_match = suffix_match

    def _blobs(self):
        prefix = f"assets/attribution-agent-szz-evaluation/version={self.version}/location=main/"
        for blob in bucket.list_blobs(prefix=prefix):
            if not blob.name.endswith("data.parquet"):
                continue
            tag = next((s for s in blob.name.split("/") if s.startswith("partition=")), "")
            token = tag.replace("partition=", "").replace(":", "|")
            if "|" not in token:
                continue
            repo_id, suffix = token.split("|", 1)
            if self.suffix_match(suffix):
                yield blob, repo_id

    def _read(self, item):
        blob, repo_id = item
        for attempt in range(4):
            try:
                table = pq.read_table(io.BytesIO(blob.download_as_bytes()), columns=self.columns)
                break
            except Exception:
                if attempt == 3:
                    raise
        # keep only the final graph output rows (one per trace), then drop the heavy table
        final = (
            pl.from_arrow(table)
            .filter(
                (pl.col("node") == "LangGraph")
                & (pl.col("event") == "node_end")
                & pl.col("parent").is_null()
                & pl.col("payload").is_not_null()
            )
            .select(["trace", "repository", "correction", "payload"])
        )
        del table
        rows = []
        for row in final.iter_rows(named=True):
            payload = json.loads(row["payload"])
            pred = ((payload.get("outputs") or {}).get("introducer") or "").lower() or None
            rows.append(
                {"repo_id": repo_id, "repository": row["repository"], "correction": row["correction"], "pred": pred}
            )
        return rows

    def load(self):
        rows = []
        targets = list(self._blobs())
        with ThreadPoolExecutor(max_workers=8) as pool:
            for batch in pool.map(self._read, targets):
                rows.extend(batch)
        gc.collect()
        return pl.DataFrame(rows)


class Bootstrap:
    """Bootstrap percentile interval for the tie-handled AUROC of a reliability score."""

    def __init__(self, ranker, reps=2000):
        self.ranker = ranker
        self.reps = reps

    def interval(self, score, labels):
        score, labels, n = np.asarray(score), np.asarray(labels), len(labels)
        state = np.random.RandomState(0)
        draws = []
        for _ in range(self.reps):
            idx = state.randint(0, n, n)
            value = self.ranker.auroc(score[idx], labels[idx])
            if value == value:  # skip degenerate resamples (all one class)
                draws.append(value)
        return np.percentile(draws, [2.5, 50, 97.5])


def selection(arm):
    """Majority prediction per corrective change, with vote share and gold correctness."""
    joined = arm.join(
        runs.filter(pl.col("language") == "py").select(["repository", "correction", "defects"]).unique(),
        on=["repository", "correction"],
        how="left",
    )
    valid = {}
    for triple in joined.select(["repo_id", "correction", "pred"]).unique().iter_rows(named=True):
        pred = triple["pred"]
        key = (triple["repo_id"], triple["correction"], pred)
        if pred is None or not re.fullmatch(r"[0-9a-f]{7,40}", pred):
            valid[key] = False
            continue
        try:
            repos[triple["repo_id"]].revparse_single(pred)
            valid[key] = True
        except Exception:
            valid[key] = False
    answered = joined.filter(
        pl.struct(["repo_id", "correction", "pred"]).map_elements(
            lambda s: valid[(s["repo_id"], s["correction"], s["pred"])], return_dtype=pl.Boolean
        )
    )
    rows = []
    for keys, group in answered.group_by(["repository", "correction"]):
        sha, votes = Counter(group["pred"].to_list()).most_common(1)[0]
        gold = group["defects"].first() or []
        rows.append({"vote": votes / group.height, "Y": int(any(sha[:12] == d[:12] for d in gold))})
    return pl.DataFrame(rows), joined.height - answered.height


bootstrap = Bootstrap(ranking)
arms = {
    "M3 greedy": TemperatureArm("1.0.1", lambda s: s in {"1-minimax-m3", "2-minimax-m3", "3-minimax-m3"}),
    "M3 hot 0.7": TemperatureArm("1.1.0", lambda s: s.endswith("-minimax-m3-hot")),
}
ablation = []
for name, arm in arms.items():
    sel, dropped = selection(arm.load())
    split = sel.filter(pl.col("vote") < 1.0)
    agree = sel.filter(pl.col("vote") == 1.0)
    lo, md, hi = bootstrap.interval(sel["vote"].to_numpy(), sel["Y"].to_numpy())
    ablation.append(
        {
            "arm": name,
            "changes": sel.height,
            "errors": dropped,
            "base acc": round(sel["Y"].mean(), 3),
            "unanimous": round((sel["vote"] == 1.0).mean(), 2),
            "agree acc": round(agree["Y"].mean(), 2),
            "split n": split.height,
            "split acc": round(split["Y"].mean(), 2),
            "vote AUROC": f"{md:.2f} [{lo:.2f}, {hi:.2f}]",
        }
    )
with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=60):
    display(pl.DataFrame(ablation))

Temperature has a measurable effect on agreement: the unanimous share falls from $0.89$ to $0.82$, so the share of split cases rises from $11\%$ to $18\%$. The vote AUROC rises from $0.66$ to $0.70$, but the bootstrap intervals overlap widely at this sample size, so the gain is not significant. Greedy decoding therefore does not degrade the agreement signal enough for a moderate temperature to carry a confirmed advantage, and point accuracy drops slightly. We keep the greedy main runs and read the split accuracy, which rests on few cases, as illustrative only. The ablation covers MiniMax M3 alone, since the hot runs exist only for it.